# MambaYOLO-B - Imagen Completa

## Configuración
- Modelo: Mamba-YOLO-B (19.1M parámetros)
- imgsz: 1600 (igual que RT-DETR y RF-DETR imagen completa)
- Batch: 2
- Épocas máx: 50
- Patience: 10
- Seed: 42
- GPU: A100 (40GB) — Colab Pro

## Entrenamiento Original

## Paso 1: Instalar condacolab

In [1]:
!pip install condacolab -q
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## Paso 2: Instalar MambaYOLO y dependencias

In [2]:
!conda create -n mambayolo python=3.11 -y
!conda run -n mambayolo pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 -q
!git clone https://github.com/HZAI-ZJNU/Mamba-YOLO.git /content/Mamba-YOLO
!conda run -n mambayolo pip install seaborn thop timm einops -q
!conda run -n mambayolo bash -c "cd /content/Mamba-YOLO/selective_scan && pip install . --no-build-isolation -q"
!conda run -n mambayolo bash -c "cd /content/Mamba-YOLO && pip install -v -e . -q"
!conda run -n mambayolo pip install "numpy<2.0" -q

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/mambayolo

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-forge

## Paso 3: Verificar instalación

In [3]:
!conda run -n mambayolo bash -c "export LD_LIBRARY_PATH=/usr/local/envs/mambayolo/lib/python3.11/site-packages/torch/lib:\$LD_LIBRARY_PATH && python -c 'import selective_scan_cuda_core; print(\"OK\")'"

OK



## Paso 4: Montar Drive y preparar dataset

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile

with zipfile.ZipFile("/content/drive/MyDrive/TFM DATA SCIENCE/dataset_kanji_prep.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/dataset_kanji_prep")

print("Descomprimido ")

Descomprimido 


In [6]:
import os
for item in os.listdir("/content/dataset_kanji_prep"):
    print(item)

dataset_kangi_prep


In [7]:
import os

ruta = "/content/dataset_kanji_prep/dataset_kangi_prep"
for item in os.listdir(ruta):
    print(item)

dataset_denoised
dataset_original


In [8]:
for split in ["train", "val", "test"]:
    n = len(os.listdir(f"/content/dataset_kanji_prep/dataset_kangi_prep/dataset_original/images/{split}"))
    print(f"{split}: {n} imágenes")

train: 55 imágenes
val: 10 imágenes
test: 10 imágenes


In [9]:
contenido_yaml = """path: /content/dataset_kanji_prep/dataset_kangi_prep/dataset_original
train: images/train
val: images/val
test: images/test

names:
  0: kanji
"""

with open("/content/dataset_kanji_prep/dataset_kangi_prep/dataset_original/dataset_original.yaml", "w") as f:
    f.write(contenido_yaml)

print("Yaml creado ")

Yaml creado 


In [10]:
contenido_yaml_denoised = """path: /content/dataset_kanji_prep/dataset_kangi_prep/dataset_denoised
train: images/train
val: images/val
test: images/test

names:
  0: kanji
"""

with open("/content/dataset_kanji_prep/dataset_kangi_prep/dataset_denoised/dataset_denoised.yaml", "w") as f:
    f.write(contenido_yaml_denoised)

print("Yaml denoised creado ")

Yaml denoised creado 


In [11]:
import os

# Verificar estructura
ruta = "/content/dataset_kanji_prep/dataset_kangi_prep/dataset_original"
print("=== Estructura ===")
for split in ["train", "val", "test"]:
    n_img = len(os.listdir(f"{ruta}/images/{split}"))
    n_lbl = len(os.listdir(f"{ruta}/labels/{split}"))
    print(f"{split}: {n_img} imágenes, {n_lbl} labels")

=== Estructura ===
train: 55 imágenes, 55 labels
val: 10 imágenes, 10 labels
test: 10 imágenes, 10 labels


## Paso 5: Entrenamiento Imagen Completa Original

In [12]:
script_original = '''
import os
os.chdir("/content/Mamba-YOLO")
import sys
sys.path.insert(0, "/content/Mamba-YOLO")
from ultralytics import YOLO
import shutil

model_conf = "/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-B.yaml"

args = {
    "data": "/content/dataset_kanji_prep/dataset_kangi_prep/dataset_original/dataset_original.yaml",
    "epochs": 50,
    "batch": 2,
    "imgsz": 1600,
    "patience": 10,
    "seed": 42,
    "device": "0",
    "amp": True,
    "project": "/content/resultados_mambayolo",
    "name": "imagen_completa_original_1600_50ep_M",
}

YOLO(model_conf).train(**args)

# Guardar resultados en Drive automáticamente
shutil.copytree(
    "/content/resultados_mambayolo/imagen_completa_original_1600_50ep_M",
    "/content/drive/MyDrive/TFM DATA SCIENCE/resultados_mambayolo/imagen_completa_original_1600_50ep_M"
)

print("Resultados guardados en Drive ")
'''

with open("/content/train_imagen_completa_original.py", "w") as f:
    f.write(script_original)

print("Script creado ")

Script creado 


In [13]:
!conda run -n mambayolo bash -c \
    "export LD_LIBRARY_PATH=/usr/local/envs/mambayolo/lib/python3.11/site-packages/torch/lib:\$LD_LIBRARY_PATH && \
    export MPLBACKEND=Agg && \
    python /content/train_imagen_completa_original.py"

WARNING ⚠️ no model scale passed. Assuming scale='B'.
New https://pypi.org/project/ultralytics/8.4.86 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.11.15 torch-2.3.0+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: task=detect, mode=train, model=/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-B.yaml, data=/content/dataset_kanji_prep/dataset_kangi_prep/dataset_original/dataset_original.yaml, epochs=50, time=None, patience=10, batch=2, imgsz=1600, save=True, save_period=-1, cache=False, device=0, workers=8, project=/content/resultados_mambayolo, name=imagen_completa_original_1600_50ep_M, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=

## Paso 5: Entrenamiento Imagen Completa denoised

In [14]:
script_denoised = '''
import os
os.chdir("/content/Mamba-YOLO")
import sys
sys.path.insert(0, "/content/Mamba-YOLO")
from ultralytics import YOLO
import shutil

model_conf = "/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-B.yaml"

args = {
    "data": "/content/dataset_kanji_prep/dataset_kangi_prep/dataset_denoised/dataset_denoised.yaml",
    "epochs": 50,
    "batch": 2,
    "imgsz": 1600,
    "patience": 10,
    "seed": 42,
    "device": "0",
    "amp": True,
    "project": "/content/resultados_mambayolo",
    "name": "imagen_completa_denoised_1600_50ep_M",
}

YOLO(model_conf).train(**args)

# Guardar resultados en Drive automáticamente
shutil.copytree(
    "/content/resultados_mambayolo/imagen_completa_denoised_1600_50ep_M",
    "/content/drive/MyDrive/TFM DATA SCIENCE/resultados_mambayolo/imagen_completa_denoised_1600_50ep_M"
)

print("Resultados guardados en Drive ")
'''

with open("/content/train_imagen_completa_denoised.py", "w") as f:
    f.write(script_denoised)

print("Script denoised creado ")

Script denoised creado 


In [15]:
!conda run -n mambayolo bash -c \
    "export LD_LIBRARY_PATH=/usr/local/envs/mambayolo/lib/python3.11/site-packages/torch/lib:\$LD_LIBRARY_PATH && \
    export MPLBACKEND=Agg && \
    python /content/train_imagen_completa_denoised.py"

WARNING ⚠️ no model scale passed. Assuming scale='B'.
New https://pypi.org/project/ultralytics/8.4.86 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.11.15 torch-2.3.0+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: task=detect, mode=train, model=/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-B.yaml, data=/content/dataset_kanji_prep/dataset_kangi_prep/dataset_denoised/dataset_denoised.yaml, epochs=50, time=None, patience=10, batch=2, imgsz=1600, save=True, save_period=-1, cache=False, device=0, workers=8, project=/content/resultados_mambayolo, name=imagen_completa_denoised_1600_50ep_M, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=